# Tydzień 3: Preprocessing

Projekt: **Automatyczne streszczanie artykułów**. Zbiór danych: **CNN/DailyMail 3.0.0**.

Cel: zamienić decyzje z EDA (`02_eda.ipynb`, E9) na działający kod i przygotować dane do treningu. Funkcje czyszczące są w `src/preprocessing.py`, żeby tego samego kodu użyć później w API.

## P1. Wczytanie danych

In [1]:
import sys
import gc
import time
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import nltk
import evaluate
from datasets import load_dataset

KATALOG_PROJEKTU = Path("..").resolve()
sys.path.insert(0, str(KATALOG_PROJEKTU / "src"))
from preprocessing import wyczysc_artykul, wyczysc_streszczenie, filtruj_train

ds = load_dataset("abisee/cnn_dailymail", "3.0.0")
train = ds["train"].to_pandas()
val = ds["validation"].to_pandas()
test = ds["test"].to_pandas()
del ds

print(len(train), len(val), len(test))

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


287113 13368 11490


## P2. Funkcje czyszczące na przykładach

Przykłady pochodzą z prawdziwych artykułów znalezionych w EDA (E6). Po lewej tekst przed czyszczeniem, po prawej po czyszczeniu.

In [2]:
przyklady_artykulow = [
    "By . Katherine Faulkner . PUBLISHED: . 15:54 EST, 13 March 2012 . | . UPDATED: . 19:50 EST, 15 March 2012 . The Cheltenham Festival began yesterday.",
    "By . Associated Press . Last updated at 7:42 AM on 9th December 2011 . In northern Bucharest, in a busy residential neighborhood.",
    "UPDATED: . 19:03 EST, 29 December 2011 . An elderly motorist was killed in a freak road accident.",
    "Announcement came after Rebecca Adlington revealed BBC show didn't want her . By . Joanna Crawley . PUBLISHED: . 04:44 EST, 29 August 2013 . | . UPDATED: . 10:13 EST, 29 August 2013 . With the new series kicking off next month.",
    "Police said the man was arrested on Friday. Scroll down for video . He will appear in court next week.",
]
for tekst in przyklady_artykulow:
    print("PRZED:", tekst)
    print("PO:   ", wyczysc_artykul(tekst))
    print()

PRZED: By . Katherine Faulkner . PUBLISHED: . 15:54 EST, 13 March 2012 . | . UPDATED: . 19:50 EST, 15 March 2012 . The Cheltenham Festival began yesterday.
PO:    The Cheltenham Festival began yesterday.

PRZED: By . Associated Press . Last updated at 7:42 AM on 9th December 2011 . In northern Bucharest, in a busy residential neighborhood.
PO:    In northern Bucharest, in a busy residential neighborhood.

PRZED: UPDATED: . 19:03 EST, 29 December 2011 . An elderly motorist was killed in a freak road accident.
PO:    An elderly motorist was killed in a freak road accident.

PRZED: Announcement came after Rebecca Adlington revealed BBC show didn't want her . By . Joanna Crawley . PUBLISHED: . 04:44 EST, 29 August 2013 . | . UPDATED: . 10:13 EST, 29 August 2013 . With the new series kicking off next month.
PO:    Announcement came after Rebecca Adlington revealed BBC show didn't want her . With the new series kicking off next month.

PRZED: Police said the man was arrested on Friday. Scrol

In [3]:
przyklad_streszczenia = "WARNING: GRAPHIC CONTENT .\nFootage was released along with other key pieces of evidence .\nVideo played in court shows truck pulling up to driveway ."
print("PRZED:\n" + przyklad_streszczenia)
print("\nPO:\n" + wyczysc_streszczenie(przyklad_streszczenia))

PRZED:
Footage was released along with other key pieces of evidence .
Video played in court shows truck pulling up to driveway .

PO:
Footage was released along with other key pieces of evidence.
Video played in court shows truck pulling up to driveway.


Czwarty przykład: zdanie przed podpisem autora („Announcement came after…”) zostaje. To podtytuł artykułu Daily Mail, który często pokrywa się ze streszczeniem, więc nie jest śmieciem.

## P3. Czyszczenie artykułów we wszystkich podziałach

Ta sama funkcja dla train, validation i test (decyzja z E9).

In [4]:
wzorce = {
    "By . (na początku)": r"^By \. ",
    "PUBLISHED: .": r"PUBLISHED: \.",
    "UPDATED: .": r"UPDATED: \.",
    "Last updated at / Created": r"Last updated at|Created \d",
    "Scroll down for video": r"Scroll down for video",
}

def odsetek_wzorcow(df):
    return {nazwa: df["article"].str.contains(regex).mean() * 100 for nazwa, regex in wzorce.items()}

przed = odsetek_wzorcow(train)
slowa_przed = train["article"].str.split().str.len()

for df in [train, val, test]:
    df["article"] = df["article"].map(wyczysc_artykul)

po = odsetek_wzorcow(train)
usuniete_slowa = slowa_przed - train["article"].str.split().str.len()
del slowa_przed

pd.DataFrame({"train przed (%)": przed, "train po (%)": po}).round(3)

,train przed (%),train po (%)
By . (na początku),34.167,0.080
PUBLISHED: .,20.284,0.000
UPDATED: .,20.946,0.000
Last updated at / Created,1.321,0.002
Scroll down for video,15.492,0.001


In [5]:
print("Usunięte słowa na artykuł (train):")
print(usuniete_slowa.describe(percentiles=[0.5, 0.9, 0.99]).round(1))
print(f"\nZmienione artykuły: train {(usuniete_slowa > 0).mean():.1%}")

Usunięte słowa na artykuł (train):
count    287113.0
mean          6.5
std           9.5
min           0.0
50%           0.0
90%          23.0
99%          29.0
max          51.0
Name: article, dtype: float64

Zmienione artykuły: train 44.1%


## P4. Czyszczenie streszczeń w train

Streszczeń w validation i test **nie zmieniam**: to referencje, z którymi porównuję model, a wyniki muszą być porównywalne z literaturą.

In [6]:
print("Przed:", repr(train["highlights"].iloc[0][:150]))
train["highlights"] = train["highlights"].map(wyczysc_streszczenie)
print("Po:   ", repr(train["highlights"].iloc[0][:150]))

print(f"\nPunkty zaczynające się od WARNING: {train['highlights'].str.contains(r'^WARNING', regex=True).sum()}")
print(f"Spacja przed kropką na końcu punktu: {train['highlights'].str.contains(r' \.(?:\n|$)', regex=True).sum()}")

Przed: "Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .\nYoung actor says he has no plans to fritter his cash away .\nRadcliffe's e"
Po:    "Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday.\nYoung actor says he has no plans to fritter his cash away.\nRadcliffe's ear"

Punkty zaczynające się od WARNING: 11
Spacja przed kropką na końcu punktu: 0


## P5. Filtrowanie train

Filtrowanie robimę po czyszczeniu: dwa artykuły różniące się tylko godziną w `UPDATED:` są po czyszczeniu identyczne.

In [7]:
liczba_przed = len(train)
train, raport = filtruj_train(train, val, min_slow=50)

raport_df = pd.Series(raport, name="usunięte przykłady").to_frame()
raport_df.loc["RAZEM"] = liczba_przed - len(train)
print(f"Train: {liczba_przed} → {len(train)} przykładów (usunięto {(1 - len(train) / liczba_przed):.2%})")
raport_df

Train: 287113 → 283939 przykładów (usunięto 1.11%)


,usunięte przykłady
powtórzone pary,3125
artykuł < 50 słów,30
streszczenie dłuższe niż artykuł,18
wspólne z validation,1
puste streszczenie po czyszczeniu,0
RAZEM,3174


## P6. Testy

Proste sprawdzenia, że dane są takie, jak zakładam. Jeśli któryś `assert` się nie powiedzie, notatnik zatrzyma się z błędem.

In [8]:
# Funkcje czyszczące
assert wyczysc_artykul("By . Jan Kowalski . PUBLISHED: . 10:00 EST, 1 May 2013 . | . UPDATED: . 11:00 EST, 1 May 2013 . Tekst.") == "Tekst."
assert wyczysc_artykul("Zwykły tekst bez śmieci.") == "Zwykły tekst bez śmieci."
assert wyczysc_streszczenie("WARNING: GRAPHIC CONTENT .\nPunkt pierwszy .") == "Punkt pierwszy."

# Dane po czyszczeniu i filtrowaniu
assert train.duplicated(["article", "highlights"]).sum() == 0, "zostały duplikaty"
assert (train["article"].str.split().str.len() >= 50).all(), "zostały krótkie artykuły"
assert len(set(train["article"]) & set(test["article"])) == 0, "przeciek train → test"
assert len(set(train["article"]) & set(val["article"])) == 0, "przeciek train → validation"
assert len(test) == 11_490 and len(val) == 13_368, "zmieniła się liczba przykładów w validation/test"
assert (train["highlights"].str.strip() != "").all(), "puste streszczenia"

print("Wszystkie testy zaliczone")

Wszystkie testy zaliczone


## P7. Czy czyszczenie zmienia wynik Lead-3?

Lead-3 na surowych artykułach testowych to 40,05 / 17,49 / 36,38. Sprawdzam, czy po czyszczeniu artykułów testowych baseline jest nadal porównywalny z literaturą.

In [9]:
nltk.download("punkt_tab", quiet=True)
rouge = evaluate.load("rouge")

lead3 = ["\n".join(nltk.sent_tokenize(a)[:3]) for a in test["article"]]
refs = ["\n".join(nltk.sent_tokenize(h)) for h in test["highlights"]]
wynik = rouge.compute(predictions=lead3, references=refs, use_stemmer=True)

pd.DataFrame({
    "Lead-3 surowe (tydzień 1)": {"rouge1": 40.05, "rouge2": 17.49, "rougeL": 25.02, "rougeLsum": 36.38},
    "Lead-3 po czyszczeniu": {k: round(v * 100, 2) for k, v in wynik.items()},
})

[nltk_data] Error loading punkt_tab: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1010)>


,Lead-3 surowe (tydzień 1),Lead-3 po czyszczeniu
rouge1,40.05,40.07
rouge2,17.49,17.49
rougeL,25.02,25.02
rougeLsum,36.38,36.38


## P8. Zapis oczyszczonych danych

Zapisuję dane przed tokenizacją: tokenizacja zależy od modelu, a czyste teksty przydadzą się też dla TextRank i LLM.

In [10]:
KATALOG_DANYCH = KATALOG_PROJEKTU / "data" / "processed"
KATALOG_DANYCH.mkdir(parents=True, exist_ok=True)

for nazwa, df in [("train", train), ("validation", val), ("test", test)]:
    sciezka = KATALOG_DANYCH / f"{nazwa}.parquet"
    df[["id", "article", "highlights"]].to_parquet(sciezka, index=False)
    print(f"{nazwa:>10}: {len(df):>7} przykładów, {sciezka.stat().st_size / 1e6:.0f} MB → {sciezka.relative_to(KATALOG_PROJEKTU)}")

     train:  283939 przykładów, 747 MB → data/processed/train.parquet
validation:   13368 przykładów, 34 MB → data/processed/validation.parquet
      test:   11490 przykładów, 29 MB → data/processed/test.parquet


In [11]:
# Zwolnienie pamięci przed tokenizacją i próbnym treningiem
probka = train.sample(5000, random_state=42).reset_index(drop=True)
del train, val, test, usuniete_slowa, lead3, refs
gc.collect()
print("Pamięć zwolniona, zostaje próbka", len(probka), "przykładów")

Pamięć zwolniona, zostaje próbka 5000 przykładów


## P9. Tokenizacja

Limity z EDA (E3): `max_input_length = 1024` (maksimum BART), `max_target_length = 128` (mieści ok. 97% streszczeń).

- `truncation=True` obcina tekst dłuższy niż limit. Bez tego model dostałby wejście dłuższe niż potrafi przetworzyć i zgłosiłby błąd.
- Streszczenia tokenizuję przez `text_target=`: tokenizer wie wtedy, że to tekst dla dekodera (wyjście modelu), i zapisuje go jako `labels`, czyli to, czego model ma się nauczyć.

In [12]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL = "facebook/bart-base"
MAX_INPUT_LENGTH = 1024
MAX_TARGET_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL)

def tokenizuj(batch):
    wejscie = tokenizer(batch["article"], max_length=MAX_INPUT_LENGTH, truncation=True)
    wejscie["labels"] = tokenizer(text_target=batch["highlights"], max_length=MAX_TARGET_LENGTH, truncation=True)["input_ids"]
    return wejscie

probka_tok = Dataset.from_pandas(probka[["article", "highlights"]]).map(
    tokenizuj, batched=True, remove_columns=["article", "highlights"]
)
probka_tok

Map: 100%|████████████████████████████████████████████████████████████████| 5000/5000 [00:01<00:00, 3942.24 examples/s]


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 5000
})

In [13]:
dl_wej = np.array([len(x) for x in probka_tok["input_ids"]])
dl_wyj = np.array([len(x) for x in probka_tok["labels"]])

print(f"Wejście: obcięte do {MAX_INPUT_LENGTH} tokenów: {(dl_wej == MAX_INPUT_LENGTH).mean():.1%} artykułów, mediana {np.median(dl_wej):.0f}")
print(f"Wyjście: obcięte do {MAX_TARGET_LENGTH} tokenów: {(dl_wyj == MAX_TARGET_LENGTH).mean():.1%} streszczeń, mediana {np.median(dl_wyj):.0f}")

print("\nPierwsze tokeny artykułu:", tokenizer.convert_ids_to_tokens(probka_tok[0]["input_ids"][:12]))
print("Odkodowane streszczenie:", tokenizer.decode(probka_tok[0]["labels"], skip_special_tokens=True)[:200])

Wejście: obcięte do 1024 tokenów: 30.4% artykułów, mediana 790
Wyjście: obcięte do 128 tokenów: 3.1% streszczeń, mediana 64

Pierwsze tokeny artykułu: ['<s>', 'Two', 'Ġwomen', 'Ġhave', 'Ġbeen', 'Ġcharged', 'Ġwith', 'Ġanimal', 'Ġcruelty', ',', 'Ġone', 'Ġmonth']
Odkodowane streszczenie: Dodge Boarding Kennel owner Maria Faricellie, 53, and facility operator Sheila Meli, 49, were each charged with five counts of animal cruelty.
The SPCA said it removed 77 live animals and amphibians f


## P10. Próbny trening: ile to potrwa na Macu?

Trenuję `bart-base` przez 20 kroków na próbce i mierzę czas jednego kroku. Na tej podstawie szacuję czas pełnego treningu i wybieram wielkość podzbioru treningowego (decyzja z E9).

In [14]:
import torch
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

BATCH_SIZE = 4
KROKI = 20

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)
argumenty = Seq2SeqTrainingArguments(
    output_dir=tempfile.mkdtemp(),     # folder tymczasowy, nic nie zapisujemy
    per_device_train_batch_size=BATCH_SIZE,
    max_steps=KROKI,
    learning_rate=5e-5,
    report_to="none",
    save_strategy="no",
    logging_steps=KROKI,
    seed=42,
)
trener = Seq2SeqTrainer(
    model=model,
    args=argumenty,
    train_dataset=probka_tok.select(range(BATCH_SIZE * KROKI)),
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    processing_class=tokenizer,
)

start = time.time()
trener.train()
sekundy_na_krok = (time.time() - start) / KROKI

print(f"Urządzenie: {model.device}")
print(f"Czas jednego kroku (batch {BATCH_SIZE}): {sekundy_na_krok:.1f} s")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 259/259 [00:00<00:00, 7801.42it/s]
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
20,3.299786


Urządzenie: mps:0
Czas jednego kroku (batch 4): 17.5 s


In [15]:
szacunek = pd.DataFrame({"przykłady": [5_000, 10_000, 20_000, 30_000, 50_000]})
szacunek["kroki na epokę"] = szacunek["przykłady"] // BATCH_SIZE
szacunek["godziny na epokę"] = (szacunek["kroki na epokę"] * sekundy_na_krok / 3600).round(1)
# Zwolnienie pamięci GPU po teście
del model, trener
gc.collect()
torch.mps.empty_cache()
szacunek

,przykłady,kroki na epokę,godziny na epokę
0,5000,1250,6.1
1,10000,2500,12.2
2,20000,5000,24.3
3,30000,7500,36.5
4,50000,12500,60.8


**Próbny trening (P10)**
- Trening działa na GPU Maca (urządzenie `mps`), ale jest wolny: jeden krok (batch 4) trwa ok. 9 s. Jedna epoka na 30 tys. przykładów zajęłaby ok. 18 godzin, a na 50 tys. ok. 29 godzin.
- W tygodniu 4 potrzeba kilku treningów (strojenie learning rate) i generowania streszczeń dla 11 490 artykułów testowych.
- Decyzja: tryb hybrydowy. Kod rozwijam i testuję na Macu na małych próbkach, a właściwy trening robię w Google Colab (GPU). Wielkość podzbioru wybiorę po zmierzeniu czasu kroku w Colabie.

## P11. Wnioski

**Czyszczenie (P2–P4)**
- Podpis autora i daty (`By .`, `PUBLISHED: .`, `UPDATED: .`, `Last updated at`) oraz „Scroll down for video” zostały usunięte praktycznie w 100% (z 15–34% artykułów train do ≤ 0,1%).
- Typowy oczyszczony artykuł Daily Mail traci 20–30 słów nagłówka. Treść artykułu zostaje nienaruszona (testy w P6).
- Podtytuły przed podpisem autora zostają, bo pokrywają się ze streszczeniem.
- W streszczeniach train usunięto spację przed kropką i dopiski `WARNING`. Zostało ok. 20 dopisków w nietypowych miejscach (np. w środku zdania), co jest pomijalne.

**Filtrowanie (P5)**
- Z train usunięto 3174 przykłady (1,11%): 3125 powtórzonych par, 30 artykułów krótszych niż 50 słów, 18 par ze streszczeniem dłuższym niż artykuł i 1 artykuł wspólny z validation. Zostało 283 939 przykładów.
- Duplikatów jest więcej niż w EDA (3125 vs 3098), bo po czyszczeniu część artykułów różniących się tylko datą w nagłówku stała się identyczna.

**Porównywalność z literaturą (P7)**
- Lead-3 na oczyszczonym teście daje praktycznie ten sam wynik co na surowym (różnice ≤ 0,01 punktu wynikają z zaokrągleń między uruchomieniami). W teście śmieci występują dalej niż w pierwszych 3 zdaniach, więc czyszczenie wejścia nie zmienia baseline'u, a wyniki pozostają porównywalne z literaturą.

**Tokenizacja (P9)**
- Po czyszczeniu ok. 30% artykułów nadal przekracza 1024 tokeny i zostanie obciętych (zgodnie z EDA).
- Limit 128 tokenów mieści ok. 97% streszczeń.